# TrafficGuard AI
## Phase 4: Data Preprocessing and Cleaning

### Objectives

- Remove exact duplicate images.
- Prevent cross-split data leakage.
- Group images originating from the same video or source.
- Investigate perceptually similar images.
- Validate and standardize YOLO annotations.
- Remap the seatbelt dataset to two target classes.
- Create independent training, validation and testing sets.
- Generate final YOLO dataset configuration files.
- Save preprocessing statistics and visualizations.

The original raw datasets remain unchanged throughout
the preprocessing process.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%pip install -q ImageHash

In [ ]:
import os
import re
import cv2
import yaml
import json
import random
import shutil
import hashlib
import imagehash

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter, defaultdict
from sklearn.model_selection import GroupShuffleSplit
from matplotlib.patches import Rectangle

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid")

plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "font.size": 11
})

print("All libraries imported successfully!")

In [ ]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

RAW_ROOT = PROJECT_ROOT / "datasets/raw"
PROCESSED_ROOT = PROJECT_ROOT / "datasets/processed"

EDA_TABLES = PROJECT_ROOT / "results/eda/tables"

PREPROCESSING_DIR = (
    PROJECT_ROOT / "results/preprocessing"
)

FIGURES_DIR = PREPROCESSING_DIR / "figures"
TABLES_DIR = PREPROCESSING_DIR / "tables"
REPORTS_DIR = PREPROCESSING_DIR / "reports"

LOCAL_ROOT = Path("/content/trafficguard_preprocessing")

LOCAL_RAW = LOCAL_ROOT / "raw"
LOCAL_PROCESSED = LOCAL_ROOT / "processed"

for folder in [
    PROCESSED_ROOT,
    FIGURES_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    LOCAL_RAW,
    LOCAL_PROCESSED
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

SOURCE_PATHS = {
    "Helmet": RAW_ROOT / "helmet/labeled_dataset",
    "Seatbelt": RAW_ROOT / "seatbelt"
}

ORIGINAL_SPLITS = {
    "Helmet": {
        "train": "train",
        "val": "val",
        "test": "test"
    },

    "Seatbelt": {
        "train": "train",
        "val": "valid",
        "test": "test"
    }
}

for dataset, path in SOURCE_PATHS.items():

    assert path.exists(), f"{dataset} dataset missing!"

    print(f"{dataset}: {path}")

print("\nPreprocessing outputs:")
print(PREPROCESSING_DIR)

print("\nProject configuration successful!")

In [ ]:
images_df = pd.read_csv(
    EDA_TABLES / "image_inventory.csv"
)

objects_df = pd.read_csv(
    EDA_TABLES / "object_annotations.csv"
)

issues_df = pd.read_csv(
    EDA_TABLES / "dataset_issues.csv"
)

duplicates_df = pd.read_csv(
    EDA_TABLES / "exact_duplicate_images.csv"
)

leakage_df = pd.read_csv(
    EDA_TABLES / "cross_split_leakage.csv"
)

near_duplicates_df = pd.read_csv(
    EDA_TABLES / "potential_near_duplicates.csv"
)

phash_df = pd.read_csv(
    EDA_TABLES / "perceptual_hashes.csv",
    dtype={"Perceptual Hash": str}
)

print("=" * 60)
print("PREVIOUS EDA RESULTS LOADED")
print("=" * 60)

print("Image records:", len(images_df))
print("Annotation records:", len(objects_df))
print("Dataset issues:", len(issues_df))
print("Exact duplicate entries:", len(duplicates_df))
print("Cross-split leakage entries:", len(leakage_df))
print("Potential near-duplicate pairs:", len(near_duplicates_df))
print("Perceptual hashes:", len(phash_df))

assert len(images_df) == 7873
assert len(objects_df) == 30979
assert images_df["Image ID"].is_unique

print("\nEDA results verified successfully!")


In [ ]:
for dataset_name, source in SOURCE_PATHS.items():

    destination = LOCAL_RAW / dataset_name.lower()

    print(f"\nCopying {dataset_name} dataset...")

    for split in ORIGINAL_SPLITS[dataset_name].values():

        for folder in ["images", "labels"]:

            src = source / split / folder
            dst = destination / split / folder

            assert src.exists(), f"Missing folder: {src}"

            shutil.copytree(
                src,
                dst,
                dirs_exist_ok=True,
                ignore=shutil.ignore_patterns(
                    ".cache",
                    "*.cache",
                    ".DS_Store"
                )
            )

            print(f"Copied: {split}/{folder}")

print("\nBoth datasets copied successfully!")

In [ ]:
def reconstruct_paths(row):

    dataset, split, relative = row["Image ID"].split(
        "/", 2
    )

    actual_split = ORIGINAL_SPLITS[dataset][split]

    base = (
        LOCAL_RAW /
        dataset.lower() /
        actual_split
    )

    relative = Path(relative)

    image_path = base / "images" / relative

    label_path = (
        base /
        "labels" /
        relative.with_suffix(".txt")
    )

    return pd.Series({
        "Current Image Path": str(image_path),
        "Current Label Path": str(label_path)
    })


new_paths = images_df.apply(
    reconstruct_paths,
    axis=1
)

images_df = pd.concat(
    [images_df, new_paths],
    axis=1
)

missing_images = images_df[
    ~images_df["Current Image Path"].map(
        lambda p: Path(p).is_file()
    )
]

missing_labels = images_df[
    ~images_df["Current Label Path"].map(
        lambda p: Path(p).is_file()
    )
]

print("Missing image files:", len(missing_images))
print("Missing annotation files:", len(missing_labels))

assert missing_images.empty
assert missing_labels.empty

print("\nAll source paths verified successfully!")

In [ ]:
print("ORIGINAL DATASET ISSUES")

display(issues_df)

issue_summary = (
    issues_df.groupby(
        ["Dataset", "Issue"]
    )
    .size()
    .reset_index(name="Count")
)

display(issue_summary)

issue_summary.to_csv(
    TABLES_DIR / "original_dataset_issues.csv",
    index=False
)

In [ ]:
CLASS_MAPPINGS = {
    "Helmet": {
        0: 0,
        1: 1
    },

    "Seatbelt": {
        0: 1,
        1: 0
    }
}

FINAL_CLASSES = {
    "Helmet": {
        0: "with_helmet",
        1: "without_helmet"
    },

    "Seatbelt": {
        0: "with_seatbelt",
        1: "without_seatbelt"
    }
}

ORIGINAL_CLASS_IDS = {
    "Helmet": {0, 1},
    "Seatbelt": {0, 1, 2, 3}
}

for dataset, mapping in CLASS_MAPPINGS.items():

    print(f"\n{dataset} class mapping:")

    for old_id, new_id in mapping.items():

        print(
            f"Original {old_id} -> "
            f"New {new_id}: "
            f"{FINAL_CLASSES[dataset][new_id]}"
        )

In [ ]:
def clean_annotation(label_path, dataset_name):

    cleaned = []
    original_valid = []
    events = []
    seen = set()
    unresolved = False

    mapping = CLASS_MAPPINGS[dataset_name]

    valid_ids = ORIGINAL_CLASS_IDS[dataset_name]

    lines = Path(label_path).read_text(
        encoding="utf-8"
    ).splitlines()

    for line_number, line in enumerate(
        lines,
        start=1
    ):

        line = line.strip()

        if not line:
            continue

        class_id = None

        try:

            parts = line.split()

            if len(parts) != 5:
                raise ValueError(
                    "Incorrect number of YOLO values"
                )

            class_value = float(parts[0])

            if not class_value.is_integer():
                raise ValueError("Non-integer class ID")

            class_id = int(class_value)

            if class_id not in valid_ids:
                raise ValueError("Unknown class ID")

            x, y, w, h = map(
                float,
                parts[1:]
            )

            if not np.isfinite(
                [x, y, w, h]
            ).all():

                raise ValueError(
                    "Non-finite coordinates"
                )

            if w <= 0 or h <= 0:
                raise ValueError(
                    "Non-positive box dimensions"
                )

            x1 = x - w / 2
            y1 = y - h / 2
            x2 = x + w / 2
            y2 = y + h / 2

            tolerance = 0.0001

            overflow = max(
                0,
                -x1,
                -y1,
                x2 - 1,
                y2 - 1
            )

            if overflow > tolerance:

                raise ValueError(
                    "Bounding box significantly outside image"
                )

            if overflow > 0:

                events.append({
                    "Line": line_number,
                    "Action": "Minor boundary correction"
                })

            x1 = max(0.0, x1)
            y1 = max(0.0, y1)
            x2 = min(1.0, x2)
            y2 = min(1.0, y2)

            w = x2 - x1
            h = y2 - y1

            if w <= 0 or h <= 0:

                raise ValueError(
                    "Degenerate bounding box"
                )

            x = (x1 + x2) / 2
            y = (y1 + y2) / 2

            original_box = (
                class_id,
                round(x, 8),
                round(y, 8),
                round(w, 8),
                round(h, 8)
            )

            original_valid.append(original_box)

            if class_id not in mapping:

                events.append({
                    "Line": line_number,
                    "Action": "Auxiliary class excluded"
                })

                continue

            new_id = mapping[class_id]

            box = (
                new_id,
                round(x, 8),
                round(y, 8),
                round(w, 8),
                round(h, 8)
            )

            if box in seen:

                events.append({
                    "Line": line_number,
                    "Action": "Duplicate annotation removed"
                })

                continue

            seen.add(box)
            cleaned.append(box)

        except Exception as error:

            if (
                dataset_name == "Seatbelt"
                and class_id in {2, 3}
            ):

                action = (
                    "Invalid auxiliary annotation excluded"
                )

            else:

                unresolved = True

                action = (
                    "Unresolved target annotation"
                )

            events.append({
                "Line": line_number,
                "Action": action,
                "Detail": str(error)
            })

    return (
        sorted(cleaned),
        events,
        unresolved,
        sorted(set(original_valid))
    )


print("Annotation cleaning function initialized.")

In [ ]:
annotation_map = {}
original_annotation_map = {}

cleaning_events = []
preprocessing_records = []

for row in tqdm(
    images_df.to_dict("records"),
    desc="Cleaning annotations"
):

    image_id = row["Image ID"]

    cleaned, events, unresolved, original = clean_annotation(
        row["Current Label Path"],
        row["Dataset"]
    )

    annotation_map[image_id] = cleaned

    original_annotation_map[image_id] = original

    for event in events:

        cleaning_events.append({
            "Dataset": row["Dataset"],
            "Image ID": image_id,
            **event
        })

    counts = Counter(
        box[0] for box in cleaned
    )

    preprocessing_records.append({
        **row,

        "Final Objects": len(cleaned),

        "Class 0 Objects": counts[0],

        "Class 1 Objects": counts[1],

        "Unresolved Annotation": unresolved,

        "Status": (
            "Review invalid annotation"
            if unresolved
            else (
                "Review empty target labels"
                if not cleaned
                else "Eligible"
            )
        )
    })

cleaned_df = pd.DataFrame(
    preprocessing_records
)

events_df = pd.DataFrame(
    cleaning_events
)

print("\nCLEANING STATUS")

display(
    cleaned_df.groupby(
        ["Dataset", "Status"]
    ).size().reset_index(name="Images")
)

print("\nANNOTATION CLEANING EVENTS")

display(
    events_df.groupby(
        ["Dataset", "Action"]
    ).size().reset_index(name="Count")
)

events_df.to_csv(
    TABLES_DIR / "annotation_cleaning_events.csv",
    index=False
)

print("\nAnnotation cleaning completed.")

In [ ]:
review_df = cleaned_df[
    cleaned_df["Status"] != "Eligible"
].copy()

display(
    review_df[
        [
            "Dataset",
            "Image ID",
            "Label Status",
            "Valid Objects",
            "Final Objects",
            "Status"
        ]
    ]
)

review_df.to_csv(
    TABLES_DIR / "annotation_review_candidates.csv",
    index=False
)

print(
    "\nTotal images requiring review:",
    len(review_df)
)

In [ ]:
for dataset_name in ["Helmet", "Seatbelt"]:

    subset = review_df[
        review_df["Dataset"] == dataset_name
    ]

    if subset.empty:

        print(f"No review candidates: {dataset_name}")
        continue

    samples = subset.head(18)

    columns = 3

    rows = int(
        np.ceil(len(samples) / columns)
    )

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(15, rows * 4),
        squeeze=False
    )

    for ax in axes.flat:
        ax.axis("off")

    for ax, (_, row) in zip(
        axes.flat,
        samples.iterrows()
    ):

        with Image.open(
            row["Current Image Path"]
        ) as img:

            ax.imshow(img.convert("RGB"))

        ax.set_title(
            f"{row['Image'][:28]}\n"
            f"{row['Status']}",
            fontsize=9
        )

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR /
        f"01_{dataset_name.lower()}_review_candidates.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"{dataset_name}: displaying "
        f"{len(samples)} of {len(subset)} candidates."
    )

In [ ]:
# Add image IDs here only after manually confirming
# that they contain no relevant target objects.

APPROVED_BACKGROUND_IDS = set()

approved_mask = (
    cleaned_df["Image ID"].isin(
        APPROVED_BACKGROUND_IDS
    )
    & ~cleaned_df["Unresolved Annotation"]
    & (cleaned_df["Final Objects"] == 0)
)

cleaned_df.loc[
    approved_mask,
    "Status"
] = "Eligible"

quarantine_df = cleaned_df[
    cleaned_df["Status"] != "Eligible"
].copy()

eligible_df = cleaned_df[
    cleaned_df["Status"] == "Eligible"
].copy()

quarantine_df.to_csv(
    TABLES_DIR / "quarantined_images.csv",
    index=False
)

print("Eligible images:", len(eligible_df))

print("Quarantined images:", len(quarantine_df))

display(
    eligible_df.groupby("Dataset")
    .size()
    .reset_index(name="Eligible Images")
)

In [ ]:
retained_records = []

duplicate_log = []

for (dataset, sha256), group in tqdm(
    cleaned_df.groupby(["Dataset", "SHA256"]),
    desc="Checking exact duplicate groups"
):

    records = group.to_dict("records")

    eligible = [
        row for row in records
        if row["Status"] == "Eligible"
    ]

    if not eligible:
        continue

    signatures = {
        tuple(
            original_annotation_map[row["Image ID"]]
        )
        for row in records
    }

    has_conflict = (
        len(signatures) > 1
        or any(
            row["Unresolved Annotation"]
            for row in records
        )
    )

    if len(records) > 1 and has_conflict:

        for row in eligible:

            duplicate_log.append({
                "Dataset": dataset,
                "Image ID": row["Image ID"],
                "SHA256": sha256,
                "Action": "Quarantined conflicting duplicate"
            })

        continue

    eligible = sorted(
        eligible,
        key=lambda row: row["Image ID"]
    )

    selected = eligible[0]

    retained_records.append(selected)

    for row in eligible[1:]:

        duplicate_log.append({
            "Dataset": dataset,
            "Image ID": row["Image ID"],
            "Retained Image ID": selected["Image ID"],
            "SHA256": sha256,
            "Action": "Redundant exact duplicate excluded"
        })

unique_df = pd.DataFrame(retained_records)

duplicate_log_df = pd.DataFrame(
    duplicate_log,
    columns=[
        "Dataset",
        "Image ID",
        "Retained Image ID",
        "SHA256",
        "Action"
    ]
)

duplicate_log_df.to_csv(
    TABLES_DIR / "duplicate_removal_log.csv",
    index=False
)

assert not unique_df.duplicated(
    subset=["Dataset", "SHA256"]
).any()

print("\nEXACT DEDUPLICATION RESULTS")

display(
    unique_df.groupby("Dataset")
    .size()
    .reset_index(name="Retained Images")
)

display(
    duplicate_log_df.groupby(
        ["Dataset", "Action"]
    ).size().reset_index(name="Images")
)

In [ ]:
def extract_source_group(dataset_name, filename):

    stem = Path(filename).stem.lower()

    if dataset_name == "Helmet":

        match = re.match(
            r"^(source\d+)[_-]frame\d+",
            stem
        )

        if match:
            return match.group(1)

        return None

    if dataset_name == "Seatbelt":

        # Strip recognized Roboflow export suffix.
        stem = re.sub(
            r"\.rf\.[a-f0-9]+$",
            "",
            stem
        )

        # Strip the original extension marker.
        stem = re.sub(
            r"[_\.](jpg|jpeg|png|webp)$",
            "",
            stem
        )

        # Recognize explicit source/frame naming.
        match = re.match(
            r"^(source\d+)[_-]frame\d+",
            stem
        )

        if match:
            return match.group(1)

        return stem


unique_df["Source Group"] = unique_df.apply(
    lambda row: extract_source_group(
        row["Dataset"],
        row["Image"]
    ),
    axis=1
)

source_summary = (
    unique_df.groupby(
        ["Dataset", "Source Group"],
        dropna=False
    )
    .size()
    .reset_index(name="Images")
)

print("HELMET SOURCE GROUPS")

display(
    source_summary[
        source_summary["Dataset"] == "Helmet"
    ].sort_values(
        "Images",
        ascending=False
    )
)

print("\nLARGEST SEATBELT SOURCE GROUPS")

display(
    source_summary[
        source_summary["Dataset"] == "Seatbelt"
    ].sort_values(
        "Images",
        ascending=False
    ).head(20)
)

source_summary.to_csv(
    TABLES_DIR / "source_group_distribution.csv",
    index=False
)

unknown_helmet = unique_df[
    (unique_df["Dataset"] == "Helmet")
    & unique_df["Source Group"].isna()
]

print(
    "\nHelmet images with unidentified sources:",
    len(unknown_helmet)
)

In [ ]:
hash_lookup = phash_df.set_index(
    "Image ID"
)["Perceptual Hash"].to_dict()

unique_df["Perceptual Hash"] = (
    unique_df["Image ID"].map(hash_lookup)
)

# Calculate any missing hashes.
missing_hash_mask = (
    unique_df["Perceptual Hash"].isna()
)

for index, row in unique_df[
    missing_hash_mask
].iterrows():

    with Image.open(
        row["Current Image Path"]
    ) as img:

        unique_df.at[
            index,
            "Perceptual Hash"
        ] = str(
            imagehash.phash(img, hash_size=8)
        )

assert unique_df["Perceptual Hash"].notna().all()

assert unique_df[
    "Perceptual Hash"
].map(
    lambda value: bool(
        re.fullmatch(
            r"[0-9a-fA-F]{16}",
            str(value)
        )
    )
).all()

print(
    "Verified perceptual hashes:",
    len(unique_df)
)

In [ ]:
class UnionFind:

    def __init__(self, n):

        self.parent = list(range(n))

        self.rank = [0] * n

    def find(self, x):

        if self.parent[x] != x:

            self.parent[x] = self.find(
                self.parent[x]
            )

        return self.parent[x]

    def union(self, a, b):

        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return

        if self.rank[root_a] < self.rank[root_b]:

            self.parent[root_a] = root_b

        elif self.rank[root_a] > self.rank[root_b]:

            self.parent[root_b] = root_a

        else:

            self.parent[root_b] = root_a

            self.rank[root_a] += 1


print("Union-Find grouping algorithm initialized.")

In [ ]:
HASH_THRESHOLD = 5

grouped_datasets = []

similarity_logs = []

for dataset_name in ["Helmet", "Seatbelt"]:

    subset = unique_df[
        unique_df["Dataset"] == dataset_name
    ].copy().reset_index(drop=True)

    if dataset_name == "Helmet":

        assert subset["Source Group"].notna().all(), (
            "Some helmet source IDs require manual review."
        )

    uf = UnionFind(len(subset))

    source_groups = defaultdict(list)

    for index, row in subset.iterrows():

        source = row["Source Group"]

        if pd.notna(source):

            source_groups[source].append(index)

    # Group images with a common source.
    for members in source_groups.values():

        for index in members[1:]:

            uf.union(
                members[0],
                index
            )

    hashes = [
        int(value, 16)
        for value in subset["Perceptual Hash"]
    ]

    ids = subset["Image ID"].tolist()

    matched_pairs = 0

    for i in tqdm(
        range(len(subset)),
        desc=f"Similarity grouping: {dataset_name}"
    ):

        hash_i = hashes[i]

        for j in range(i + 1, len(subset)):

            distance = (
                hash_i ^ hashes[j]
            ).bit_count()

            if distance <= HASH_THRESHOLD:

                uf.union(i, j)

                matched_pairs += 1

                similarity_logs.append({
                    "Dataset": dataset_name,
                    "Image A": ids[i],
                    "Image B": ids[j],
                    "Hamming Distance": distance
                })

    roots = [
        uf.find(i)
        for i in range(len(subset))
    ]

    root_mapping = {
        root: f"{dataset_name.lower()}_group_{i:05d}"

        for i, root in enumerate(
            sorted(set(roots))
        )
    }

    subset["Group ID"] = [
        root_mapping[root]
        for root in roots
    ]

    grouped_datasets.append(subset)

    print(f"\n{dataset_name}")

    print("Images:", len(subset))

    print(
        "Source groups:",
        subset["Source Group"].nunique()
    )

    print(
        "Final connected groups:",
        subset["Group ID"].nunique()
    )

    print(
        "Similar image pairs:",
        matched_pairs
    )


grouped_df = pd.concat(
    grouped_datasets,
    ignore_index=True
)

similarity_df = pd.DataFrame(
    similarity_logs,
    columns=[
        "Dataset",
        "Image A",
        "Image B",
        "Hamming Distance"
    ]
)

similarity_df.to_csv(
    TABLES_DIR / "similarity_grouping_pairs.csv",
    index=False
)

In [ ]:
group_summary = (
    grouped_df.groupby(
        ["Dataset", "Group ID"]
    )
    .agg(
        Images=("Image ID", "size"),

        Source_Groups=(
            "Source Group",
            "nunique"
        ),

        Class_0_Objects=(
            "Class 0 Objects",
            "sum"
        ),

        Class_1_Objects=(
            "Class 1 Objects",
            "sum"
        )
    )
    .reset_index()
)

group_summary.to_csv(
    TABLES_DIR / "connected_group_summary.csv",
    index=False
)

for dataset_name in ["Helmet", "Seatbelt"]:

    subset = group_summary[
        group_summary["Dataset"] == dataset_name
    ]

    total_images = subset["Images"].sum()

    largest = subset["Images"].max()

    print("\n", dataset_name)

    print("Connected groups:", len(subset))

    print("Largest group:", largest)

    print(
        "Largest group percentage:",
        round(largest / total_images * 100, 2)
    )

    display(
        subset.sort_values(
            "Images",
            ascending=False
        ).head(10)
    )

In [ ]:
path_lookup = grouped_df.set_index(
    "Image ID"
)["Current Image Path"].to_dict()

for dataset_name in ["Helmet", "Seatbelt"]:

    subset = similarity_df[
        similarity_df["Dataset"] == dataset_name
    ]

    if subset.empty:

        print(f"No similarity pairs: {dataset_name}")

        continue

    samples = subset.sample(
        n=min(8, len(subset)),
        random_state=SEED
    )

    fig, axes = plt.subplots(
        len(samples),
        2,
        figsize=(12, 3.5 * len(samples)),
        squeeze=False
    )

    for i, (_, row) in enumerate(
        samples.iterrows()
    ):

        for j, column in enumerate(
            ["Image A", "Image B"]
        ):

            with Image.open(
                path_lookup[row[column]]
            ) as img:

                axes[i, j].imshow(
                    img.convert("RGB")
                )

            axes[i, j].axis("off")

        axes[i, 0].set_title("Image A")

        axes[i, 1].set_title(
            f"Image B — Distance: "
            f"{row['Hamming Distance']}"
        )

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR /
        f"02_{dataset_name.lower()}_similarity_review.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
TARGET_RATIOS = np.array([
    0.70,
    0.15,
    0.15
])

SPLIT_NAMES = [
    "train",
    "val",
    "test"
]

def create_group_split(df, trials=2000):

    groups = (
        df.groupby("Group ID")
        .agg(
            Images=("Image ID", "size"),

            Class0=("Class 0 Objects", "sum"),

            Class1=("Class 1 Objects", "sum")
        )
        .reset_index()
    )

    values = groups[
        ["Images", "Class0", "Class1"]
    ].to_numpy(dtype=float)

    totals = values.sum(axis=0)

    n_groups = len(groups)

    if n_groups < 3:

        raise ValueError(
            "Insufficient independent groups."
        )

    if np.any(totals == 0):

        raise ValueError(
            "A required target class is missing."
        )

    rng = np.random.default_rng(SEED)

    best_score = np.inf
    best_assignment = None

    largest_group = values[:, 0].max()

    print("Available groups:", n_groups)

    print("Largest group:", int(largest_group))

    for trial in range(trials):

        assignment = rng.choice(
            3,
            size=n_groups,
            p=TARGET_RATIOS
        )

        split_totals = np.zeros(
            (3, 3),
            dtype=float
        )

        np.add.at(
            split_totals,
            assignment,
            values
        )

        # All splits must contain images and both classes.
        if np.any(split_totals == 0):
            continue

        proportions = (
            split_totals /
            totals[np.newaxis, :]
        )

        image_error = np.sum(
            (
                proportions[:, 0]
                - TARGET_RATIOS
            ) ** 2
        )

        class_error = np.sum(
            (
                proportions[:, 1:]
                - TARGET_RATIOS[:, np.newaxis]
            ) ** 2
        )

        image_proportions = proportions[:, 0]

        # Prevent extremely small evaluation sets.
        if (
            image_proportions[0] < 0.50
            or image_proportions[1] < 0.10
            or image_proportions[2] < 0.10
        ):

            continue

        score = (
            3 * image_error
            + class_error
        )

        if score < best_score:

            best_score = score

            best_assignment = (
                assignment.copy()
            )

    if best_assignment is None:

        raise ValueError(
            "No acceptable group-based split found. "
            "Review source groups and similarity matches."
        )

    mapping = {
        group: SPLIT_NAMES[split_index]

        for group, split_index in zip(
            groups["Group ID"],
            best_assignment
        )
    }

    result = df.copy()

    result["Final Split"] = result[
        "Group ID"
    ].map(mapping)

    print("Best split score:", best_score)

    return result


print("Group-aware splitting function initialized.")

In [ ]:
final_datasets = []

for dataset_name in ["Helmet", "Seatbelt"]:

    print("\n" + "=" * 55)

    print(
        f"CREATING {dataset_name.upper()} SPLITS"
    )

    print("=" * 55)

    subset = grouped_df[
        grouped_df["Dataset"] == dataset_name
    ].copy()

    final = create_group_split(
        subset,
        trials=2000
    )

    final_datasets.append(final)

final_df = pd.concat(
    final_datasets,
    ignore_index=True
)

final_split_summary = (
    final_df.groupby(
        ["Dataset", "Final Split"]
    )
    .agg(
        Images=("Image ID", "size"),

        Groups=("Group ID", "nunique"),

        Class_0_Objects=(
            "Class 0 Objects",
            "sum"
        ),

        Class_1_Objects=(
            "Class 1 Objects",
            "sum"
        )
    )
    .reset_index()
)

final_split_summary["Image_Percentage"] = (
    final_split_summary["Images"] /
    final_split_summary.groupby(
        "Dataset"
    )["Images"].transform("sum")
    * 100
).round(2)

display(final_split_summary)

final_split_summary.to_csv(
    TABLES_DIR / "final_dataset_splits.csv",
    index=False
)

final_df.to_csv(
    TABLES_DIR / "final_split_manifest.csv",
    index=False
)

In [ ]:
for dataset_name in ["Helmet", "Seatbelt"]:

    subset = final_df[
        final_df["Dataset"] == dataset_name
    ]

    group_leakage = (
        subset.groupby("Group ID")[
            "Final Split"
        ].nunique() > 1
    ).sum()

    exact_leakage = (
        subset.groupby("SHA256")[
            "Final Split"
        ].nunique() > 1
    ).sum()

    source_leakage = (
        subset.groupby("Source Group")[
            "Final Split"
        ].nunique() > 1
    ).sum()

    class_counts = (
        subset.groupby("Final Split")[
            [
                "Class 0 Objects",
                "Class 1 Objects"
            ]
        ].sum()
    )

    print("\n", dataset_name)

    print("Group leakage:", group_leakage)

    print("Exact image leakage:", exact_leakage)

    print("Recognized source leakage:", source_leakage)

    display(class_counts)

    assert group_leakage == 0

    assert exact_leakage == 0

    assert source_leakage == 0

    assert set(
        subset["Final Split"]
    ) == {"train", "val", "test"}

    assert (class_counts > 0).all().all()

print("\nAll proposed split checks passed!")

In [ ]:
import tempfile

EXPORT_ROOT = Path(
    tempfile.mkdtemp(
        prefix="trafficguard_processed_",
        dir=str(LOCAL_PROCESSED)
    )
)

for dataset_name in ["helmet", "seatbelt"]:

    for split in ["train", "val", "test"]:

        for folder in ["images", "labels"]:

            path = (
                EXPORT_ROOT /
                dataset_name /
                split /
                folder
            )

            path.mkdir(
                parents=True,
                exist_ok=True
            )

print("Final dataset directories created.")

print("Export location:", EXPORT_ROOT)

In [ ]:
export_records = []

for row in tqdm(
    final_df.to_dict("records"),
    desc="Exporting processed datasets"
):

    dataset_name = row["Dataset"].lower()

    split = row["Final Split"]

    source_image = Path(
        row["Current Image Path"]
    )

    image_id = row["Image ID"]

    # Use a unique identifier to prevent filename collisions.
    unique_name = hashlib.sha256(
        image_id.encode("utf-8")
    ).hexdigest()[:20]

    filename = (
        unique_name +
        source_image.suffix.lower()
    )

    image_destination = (
        EXPORT_ROOT /
        dataset_name /
        split /
        "images" /
        filename
    )

    label_destination = (
        EXPORT_ROOT /
        dataset_name /
        split /
        "labels" /
        f"{unique_name}.txt"
    )

    assert not image_destination.exists()

    shutil.copy2(
        source_image,
        image_destination
    )

    annotations = annotation_map[image_id]

    label_lines = []

    for class_id, x, y, w, h in annotations:

        label_lines.append(
            f"{class_id} "
            f"{x:.8f} "
            f"{y:.8f} "
            f"{w:.8f} "
            f"{h:.8f}"
        )

    label_destination.write_text(
        "\n".join(label_lines),
        encoding="utf-8"
    )

    export_records.append({
        "Dataset": row["Dataset"],

        "Original Image ID": image_id,

        "Original Split": row["Split"],

        "Final Split": split,

        "Group ID": row["Group ID"],

        "Source Group": row["Source Group"],

        "Final Image": filename,

        "Relative Image Path": str(
            image_destination.relative_to(
                EXPORT_ROOT
            )
        ),

        "Relative Label Path": str(
            label_destination.relative_to(
                EXPORT_ROOT
            )
        ),

        "SHA256": row["SHA256"],

        "Class 0 Objects": row["Class 0 Objects"],

        "Class 1 Objects": row["Class 1 Objects"]
    })

export_df = pd.DataFrame(export_records)

export_df.to_csv(
    TABLES_DIR / "processed_dataset_manifest.csv",
    index=False
)

print("\nDATASET EXPORT COMPLETED")

print("Exported images:", len(export_df))

display(
    export_df.groupby(
        ["Dataset", "Final Split"]
    ).size().reset_index(name="Images")
)

In [ ]:
for dataset_name in ["Helmet", "Seatbelt"]:

    dataset_root = (
        EXPORT_ROOT /
        dataset_name.lower()
    )

    config = {
        "path": str(dataset_root),

        "train": "train/images",

        "val": "val/images",

        "test": "test/images",

        "nc": 2,

        "names": FINAL_CLASSES[dataset_name]
    }

    yaml_path = dataset_root / "data.yaml"

    with open(
        yaml_path,
        "w",
        encoding="utf-8"
    ) as file:

        yaml.safe_dump(
            config,
            file,
            sort_keys=False
        )

    print("\n", dataset_name.upper())

    print(
        yaml_path.read_text()
    )

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp"
}

validation_records = []
validation_errors = []

for dataset_name in ["Helmet", "Seatbelt"]:

    root = (
        EXPORT_ROOT /
        dataset_name.lower()
    )

    for split in ["train", "val", "test"]:

        image_dir = root / split / "images"

        label_dir = root / split / "labels"

        images = sorted([
            p for p in image_dir.iterdir()

            if p.is_file()

            and p.suffix.lower() in IMAGE_EXTENSIONS
        ])

        labels = list(
            label_dir.glob("*.txt")
        )

        counts = Counter()

        empty_labels = 0

        expected_labels = {
            p.stem for p in images
        }

        actual_labels = {
            p.stem for p in labels
        }

        if expected_labels != actual_labels:

            validation_errors.append(
                f"{dataset_name}/{split}: "
                "Image-label pairing mismatch"
            )

        for image_path in tqdm(
            images,
            desc=f"Validating {dataset_name}/{split}"
        ):

            label_path = (
                label_dir /
                f"{image_path.stem}.txt"
            )

            try:

                with Image.open(image_path) as img:

                    img.verify()

                lines = [
                    line.strip()

                    for line in label_path.read_text(
                        encoding="utf-8"
                    ).splitlines()

                    if line.strip()
                ]

                if not lines:

                    empty_labels += 1

                seen = set()

                for line in lines:

                    parts = line.split()

                    assert len(parts) == 5

                    assert parts[0] in {"0", "1"}

                    class_id = int(parts[0])

                    x, y, w, h = map(
                        float,
                        parts[1:]
                    )

                    assert np.isfinite(
                        [x, y, w, h]
                    ).all()

                    assert 0 <= x <= 1
                    assert 0 <= y <= 1
                    assert 0 < w <= 1
                    assert 0 < h <= 1

                    tolerance = 1e-7

                    assert x - w / 2 >= -tolerance
                    assert y - h / 2 >= -tolerance
                    assert x + w / 2 <= 1 + tolerance
                    assert y + h / 2 <= 1 + tolerance

                    box = (
                        class_id, x, y, w, h
                    )

                    assert box not in seen

                    seen.add(box)

                    counts[class_id] += 1

            except Exception as error:

                validation_errors.append(
                    f"{image_path}: {repr(error)}"
                )

        validation_records.append({
            "Dataset": dataset_name,

            "Split": split,

            "Images": len(images),

            "Labels": len(labels),

            "Class 0 Objects": counts[0],

            "Class 1 Objects": counts[1],

            "Empty Labels": empty_labels
        })

validation_df = pd.DataFrame(
    validation_records
)

display(validation_df)

print(
    "\nValidation errors:",
    len(validation_errors)
)

if validation_errors:

    for error in validation_errors[:20]:

        print(error)

validation_df.to_csv(
    TABLES_DIR / "processed_dataset_validation.csv",
    index=False
)

assert not validation_errors

assert validation_df["Images"].sum() == len(export_df)

assert (
    validation_df["Images"] ==
    validation_df["Labels"]
).all()

assert (
    validation_df[
        ["Class 0 Objects", "Class 1 Objects"]
    ] > 0
).all().all()

print("\nAll exported dataset files passed validation!")

In [ ]:
def compute_sha256(path):

    sha = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(65536),
            b""
        ):

            sha.update(chunk)

    return sha.hexdigest()


verified_hashes = []

for row in tqdm(
    export_df.to_dict("records"),
    desc="Verifying exported image hashes"
):

    path = (
        EXPORT_ROOT /
        row["Relative Image Path"]
    )

    current_hash = compute_sha256(path)

    assert current_hash == row["SHA256"]

    verified_hashes.append(current_hash)

export_df["Verified SHA256"] = verified_hashes

exact_duplicates = export_df.duplicated(
    subset=["Dataset", "Verified SHA256"]
).sum()

exact_cross_split = (
    export_df.groupby(
        ["Dataset", "Verified SHA256"]
    )["Final Split"].nunique() > 1
).sum()

print("Remaining exact duplicates:", exact_duplicates)

print("Exact cross-split leakage:", exact_cross_split)

assert exact_duplicates == 0

assert exact_cross_split == 0

print("\nExact duplicate validation passed!")

In [ ]:
split_lookup = final_df.set_index(
    "Image ID"
)["Final Split"].to_dict()

remaining_similarity_pairs = []

for row in similarity_df.to_dict("records"):

    split_a = split_lookup[row["Image A"]]

    split_b = split_lookup[row["Image B"]]

    if split_a != split_b:

        remaining_similarity_pairs.append({
            **row,

            "Final Split A": split_a,

            "Final Split B": split_b
        })

print(
    "Remaining detected cross-split similarity pairs:",
    len(remaining_similarity_pairs)
)

assert len(remaining_similarity_pairs) == 0

print(
    "\nAll identified similarity groups "
    "remain within individual splits."
)

In [ ]:
comparison_records = []

for dataset_name in ["Helmet", "Seatbelt"]:

    original = images_df[
        images_df["Dataset"] == dataset_name
    ]

    final = final_df[
        final_df["Dataset"] == dataset_name
    ]

    duplicates_before = duplicates_df[
        duplicates_df["Dataset"] == dataset_name
    ]["SHA256"].nunique()

    original_target_objects = objects_df[
        (objects_df["Dataset"] == dataset_name)

        & (objects_df["Class ID"].isin([0, 1]))
    ]

    comparison_records.append({
        "Dataset": dataset_name,

        "Original Images": len(original),

        "Processed Images": len(final),

        "Excluded Images": (
            len(original) - len(final)
        ),

        "Retention Percentage": round(
            len(final) / len(original) * 100,
            2
        ),

        "Original Target Objects": len(
            original_target_objects
        ),

        "Final Target Objects": int(
            final["Final Objects"].sum()
        ),

        "Original Classes": (
            2 if dataset_name == "Helmet" else 4
        ),

        "Final Classes": 2,

        "Original Duplicate Groups": duplicates_before,

        "Remaining Duplicate Groups": int(
            final.duplicated(
                subset=["SHA256"]
            ).sum()
        ),

        "Independent Groups": final["Group ID"].nunique()
    })

comparison_df = pd.DataFrame(
    comparison_records
)

display(comparison_df.T)

comparison_df.to_csv(
    TABLES_DIR / "before_after_preprocessing.csv",
    index=False
)

In [ ]:
plot_df = comparison_df.melt(
    id_vars=["Dataset"],

    value_vars=[
        "Original Images",
        "Processed Images"
    ],

    var_name="Stage",

    value_name="Images"
)

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=plot_df,

    x="Dataset",

    y="Images",

    hue="Stage",

    palette=[
        "#64748b",
        "#2563eb"
    ]
)

for container in ax.containers:

    ax.bar_label(
        container,
        padding=4,
        fmt="%d"
    )

plt.title(
    "Dataset Size Before and After Preprocessing"
)

plt.ylabel("Number of Images")

plt.xlabel("Dataset")

plt.ylim(
    0,
    plot_df["Images"].max() * 1.15
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "03_before_after_preprocessing.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 6)
)

for ax, dataset_name in zip(
    axes,
    ["Helmet", "Seatbelt"]
):

    subset = final_split_summary[
        final_split_summary["Dataset"] == dataset_name
    ].set_index(
        "Final Split"
    ).reindex(
        ["train", "val", "test"]
    )

    ax.pie(
        subset["Images"],

        labels=[
            "Training",
            "Validation",
            "Testing"
        ],

        autopct="%1.1f%%",

        startangle=90,

        colors=[
            "#2563eb",
            "#16a34a",
            "#f59e0b"
        ],

        wedgeprops={
            "edgecolor": "white",
            "linewidth": 2
        }
    )

    ax.set_title(
        f"{dataset_name} — Final Dataset Split\n"
        f"{subset['Images'].sum():,} Images"
    )

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "04_final_dataset_splits.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
balance_records = []

for dataset_name in ["Helmet", "Seatbelt"]:

    train = final_df[
        (final_df["Dataset"] == dataset_name)

        & (final_df["Final Split"] == "train")
    ]

    count0 = train["Class 0 Objects"].sum()

    count1 = train["Class 1 Objects"].sum()

    total = count0 + count1

    ratio = (
        max(count0, count1) /
        min(count0, count1)
    )

    balance_records.append({
        "Dataset": dataset_name,

        "Compliant Objects": int(count0),

        "Violation Objects": int(count1),

        "Compliant Percentage": round(
            count0 / total * 100,
            2
        ),

        "Violation Percentage": round(
            count1 / total * 100,
            2
        ),

        "Imbalance Ratio": round(ratio, 2)
    })

final_balance_df = pd.DataFrame(
    balance_records
)

display(final_balance_df)

final_balance_df.to_csv(
    TABLES_DIR / "final_training_class_balance.csv",
    index=False
)

plot_df = final_balance_df.melt(
    id_vars=["Dataset"],

    value_vars=[
        "Compliant Objects",
        "Violation Objects"
    ],

    var_name="Class",

    value_name="Objects"
)

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=plot_df,

    x="Dataset",

    y="Objects",

    hue="Class",

    palette=[
        "#16a34a",
        "#dc2626"
    ]
)

for container in ax.containers:

    ax.bar_label(
        container,
        padding=4,
        fmt="%d"
    )

plt.title(
    "Final Training Dataset Class Distribution"
)

plt.ylim(
    0,
    plot_df["Objects"].max() * 1.15
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "05_final_training_class_balance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
def show_processed_samples(dataset_name, count=9):

    subset = export_df[
        (export_df["Dataset"] == dataset_name)

        & (export_df["Final Split"] == "train")
    ]

    samples = subset.sample(
        n=min(count, len(subset)),

        random_state=SEED
    )

    fig, axes = plt.subplots(
        3, 3,
        figsize=(16, 12)
    )

    for ax in axes.flat:

        ax.axis("off")

    for ax, (_, row) in zip(
        axes.flat,
        samples.iterrows()
    ):

        image_path = (
            EXPORT_ROOT /
            row["Relative Image Path"]
        )

        label_path = (
            EXPORT_ROOT /
            row["Relative Label Path"]
        )

        with Image.open(image_path) as img:

            image = img.convert("RGB")

        width, height = image.size

        ax.imshow(image)

        for line in label_path.read_text().splitlines():

            if not line.strip():
                continue

            class_id, x, y, w, h = map(
                float,
                line.split()
            )

            class_id = int(class_id)

            xmin = (x - w / 2) * width

            ymin = (y - h / 2) * height

            color = (
                "#16a34a"
                if class_id == 0
                else "#dc2626"
            )

            rect = Rectangle(
                (xmin, ymin),

                w * width,

                h * height,

                edgecolor=color,

                facecolor="none",

                linewidth=2
            )

            ax.add_patch(rect)

            ax.text(
                xmin,

                max(0, ymin - 3),

                FINAL_CLASSES[dataset_name][class_id],

                fontsize=8,

                color="white",

                bbox={
                    "facecolor": color,
                    "edgecolor": "none",
                    "alpha": 0.9
                }
            )

        ax.set_title(
            f"{dataset_name} — Training",
            fontsize=10
        )

        ax.axis("off")

    plt.tight_layout()

    plt.savefig(
        FIGURES_DIR /
        f"06_{dataset_name.lower()}_processed_samples.png",

        dpi=300,

        bbox_inches="tight"
    )

    plt.show()


show_processed_samples("Helmet")

show_processed_samples("Seatbelt")

In [ ]:
DRIVE_VERSION = PROCESSED_ROOT / "v1"

if DRIVE_VERSION.exists():

    raise FileExistsError(
        "Processed dataset v1 already exists. "
        "Inspect it before overwriting or "
        "choose a new version."
    )

assert not validation_errors

assert exact_duplicates == 0

assert exact_cross_split == 0

assert len(remaining_similarity_pairs) == 0

shutil.copytree(
    EXPORT_ROOT,
    DRIVE_VERSION
)

# Update YAML paths for permanent Drive storage.
for dataset_name in ["helmet", "seatbelt"]:

    dataset_root = (
        DRIVE_VERSION /
        dataset_name
    )

    yaml_path = (
        dataset_root /
        "data.yaml"
    )

    config = yaml.safe_load(
        yaml_path.read_text()
    )

    config["path"] = str(dataset_root)

    yaml_path.write_text(
        yaml.safe_dump(
            config,
            sort_keys=False
        ),
        encoding="utf-8"
    )

print("Processed datasets saved permanently!")

print("\nDataset location:")

print(DRIVE_VERSION)

In [ ]:
ARCHIVE_DIR = (
    PROCESSED_ROOT /
    "archives"
)

ARCHIVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for dataset_name in ["helmet", "seatbelt"]:

    archive_path = shutil.make_archive(

        base_name=str(
            ARCHIVE_DIR /
            f"{dataset_name}_v1"
        ),

        format="zip",

        root_dir=str(EXPORT_ROOT),

        base_dir=dataset_name
    )

    archive_path = Path(archive_path)

    size_mb = (
        archive_path.stat().st_size /
        (1024 ** 2)
    )

    print(
        f"{archive_path.name}: "
        f"{size_mb:.2f} MB"
    )

print("\nTraining dataset archives created.")

In [ ]:
report_lines = [
    "TRAFFICGUARD AI",
    "PHASE 4 - DATA PREPROCESSING REPORT",
    "=" * 65,
    "",
    "PREPROCESSING CONFIGURATION",
    "-" * 65,

    f"Random seed: {SEED}",

    f"Perceptual hash threshold: {HASH_THRESHOLD}",

    "Target split: 70% train, 15% val, 15% test",

    "Splitting method: Connected source groups",

    "Exact duplicates: One consistent copy retained",

    "Conflicting annotations: Quarantined",

    "Unverified empty target labels: Quarantined",

    "Original datasets: Preserved unchanged",

    "",

    "DATASET COMPARISON",
    "-" * 65,

    comparison_df.to_string(index=False),

    "",

    "FINAL DATASET SPLITS",
    "-" * 65,

    final_split_summary.to_string(index=False),

    "",

    "FINAL TRAINING CLASS BALANCE",
    "-" * 65,

    final_balance_df.to_string(index=False),

    "",

    "FINAL DATASET VALIDATION",
    "-" * 65,

    validation_df.to_string(index=False),

    "",

    f"Remaining exact duplicates: {exact_duplicates}",

    f"Exact cross-split leakage: {exact_cross_split}",

    "Detected cross-split similarity pairs: "
    + str(len(remaining_similarity_pairs)),

    f"Validation errors: {len(validation_errors)}",

    "",

    "LIMITATIONS",
    "-" * 65,

    "Perceptual hashing does not identify every "
    "possible source relationship.",

    "Filename-derived source groups may not capture "
    "all original video identities.",

    "Final annotation semantics require visual review.",

    "",

    "NEXT PHASE",
    "-" * 65,

    "Helmet baseline model training using YOLOv8."
]

report_text = "\n".join(report_lines)

report_path = (
    REPORTS_DIR /
    "preprocessing_report.txt"
)

report_path.write_text(
    report_text,
    encoding="utf-8"
)

print(report_text)

print("\nReport saved at:")

print(report_path)

In [ ]:
print("=" * 65)

print("TRAFFICGUARD AI - PHASE 4 OUTPUTS")

print("=" * 65)

for folder in [
    FIGURES_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    ARCHIVE_DIR
]:

    print(f"\n{folder.name.upper()}")

    files = sorted([
        p for p in folder.iterdir()
        if p.is_file()
    ])

    for file in files:

        print(
            f" - {file.name} "
            f"({file.stat().st_size / 1024:.1f} KB)"
        )

    print("Total files:", len(files))

print("\n" + "=" * 65)

print("PREPARED DATASET VERIFICATION")

for dataset_name in ["helmet", "seatbelt"]:

    dataset_root = (
        DRIVE_VERSION /
        dataset_name
    )

    expected = export_df[
        export_df["Dataset"].str.lower()
        == dataset_name
    ]

    total_images = 0

    for split in ["train", "val", "test"]:

        image_dir = (
            dataset_root /
            split /
            "images"
        )

        label_dir = (
            dataset_root /
            split /
            "labels"
        )

        images = list(image_dir.iterdir())

        labels = list(label_dir.glob("*.txt"))

        expected_count = len(
            expected[
                expected["Final Split"] == split
            ]
        )

        assert len(images) == expected_count

        assert len(labels) == expected_count

        total_images += len(images)

    assert (
        dataset_root / "data.yaml"
    ).exists()

    print(
        f"{dataset_name.capitalize()}: "
        f"{total_images} verified images"
    )

print("\n" + "=" * 65)

print("PHASE 4 AUTOMATED VALIDATION PASSED")

print("Final visual review required before training.")

print("=" * 65)